# 04d - QC 済み細胞・遺伝子の original-scale merge

このノートブックで保存する **merged_qc_original_scale は、前処理を揃えた行列ではない。**
QC で選ばれた細胞・遺伝子だけを merge したものであり、発現値は各 dataset の元の scale を保持している。

```
raw_count_like の細胞では .X はraw countのまま。
cpm_tpm_like の細胞では .X はCPM/TPM-like valueのまま。
log_normalized_like の細胞では .X はlog normalized valueのまま。
```

このため、merged_qc_original_scale 全体に対して同じ意味で total_counts や pct_counts_mt を解釈してはいけない。
raw count だけを使いたい場合は `qc_preprocessing_state == "raw_count_like"` で subset する。
全データを探索的に使いたい場合は、別途 log-expression に揃える関数を使う（このノートブック末尾の関数例を参照）。

## このノートブックがやること / やらないこと

- やること: 04a〜04c で保存した `*.stage1_filtered.h5ad` を読み込み、`ad.concat` で merge する。
- **やらないこと: `sc.pp.normalize_total` / `sc.pp.log1p` / `sc.pp.scale` は実行しない。**
  したがって merge 後の `.X` は **混在スケール**である。

このmergeは「前処理を揃えた解析用merge」ではなく、**QC済み細胞・遺伝子を集めたoriginal-scale merge**である。

## 入出力

入力:
```
data/qc_h5ad/raw_count_like/*.stage1_filtered.h5ad
data/qc_h5ad/cpm_tpm_like/*.stage1_filtered.h5ad
data/qc_h5ad/log_normalized_like/*.stage1_filtered.h5ad
```

出力:
```
data/merged_h5ad/merged_qc_original_scale_outer.h5ad
data/merged_h5ad/merged_qc_original_scale_inner.h5ad
```


## セットアップ（パス・定数）

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# v2 ルートを探す（config/dataset_manifest.yaml がある場所）
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

QC_BASE = ROOT / "data" / "qc_h5ad"
STATES = ["raw_count_like", "cpm_tpm_like", "log_normalized_like"]
MERGED_DIR = ROOT / "data" / "merged_h5ad"
RESULTS_DIR = ROOT / "results" / "qc_original_scale_pipeline" / "04d_merged_original_scale"
PLOT_DIR = RESULTS_DIR / "plots"

for d in (MERGED_DIR, RESULTS_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

MAX_PLOT = 100_000
RNG = np.random.default_rng(0)

print("project root:", ROOT)
print("qc base     :", QC_BASE)
print("merged dir  :", MERGED_DIR)
print("results dir :", RESULTS_DIR)


## 04a〜04c の stage1_filtered.h5ad を読み込む

In [ ]:
# merge 後 obs に最低限残す列
OBS_KEEP = [
    "cell_uid", "source_accession", "dataset_id", "Condition",
    "qc_preprocessing_state", "qc_pass_stage1", "qc_reason_stage1",
]


def on_symbol_upper(a):
    """var_names を gene_symbol_upper に揃える（04 の merge と同じ方針）。"""
    b = a.copy()
    if "gene_symbol_upper" in b.var.columns:
        b.var_names = pd.Index(b.var["gene_symbol_upper"].astype(str))
    b.var_names_make_unique()
    return b


adatas = []
for state in STATES:
    sdir = QC_BASE / state
    files = sorted(sdir.glob("*.stage1_filtered.h5ad"))
    print(f"{state}: {len(files)} filtered h5ad")
    for p in files:
        a = sc.read_h5ad(p)
        if "qc_preprocessing_state" not in a.obs.columns:
            a.obs["qc_preprocessing_state"] = state
        keep = [c for c in OBS_KEEP if c in a.obs.columns]
        a.obs = a.obs[keep].copy()
        adatas.append(on_symbol_upper(a))

print("\ntotal adatas to merge:", len(adatas))


## merge（normalize / log1p / scale はしない → 混在スケール）

In [ ]:
# 重要: ここでは normalize_total / log1p / scale を実行しない。
# merge 後の .X は混在スケール（raw count / CPM-TPM-like / log-normalized）になる。
#
# obs_names はすでに cell_uid である前提なので index_unique=None（二重 prefix しない）。

merged_outer = ad.concat(adatas, join="outer", merge="same", index_unique=None)
merged_inner = ad.concat(adatas, join="inner", merge="same", index_unique=None)

# cell_uid は GSE をまたいで一意のはずだが念のため確認
if not merged_outer.obs_names.is_unique:
    print("WARNING: obs_names (cell_uid) is not unique after merge!")

print("merged_outer:", merged_outer.shape)
print("merged_inner:", merged_inner.shape)
print(merged_outer.obs["qc_preprocessing_state"].value_counts())


## original-scale merge を保存

In [ ]:
outer_path = MERGED_DIR / "merged_qc_original_scale_outer.h5ad"
inner_path = MERGED_DIR / "merged_qc_original_scale_inner.h5ad"
merged_outer.write_h5ad(outer_path)
merged_inner.write_h5ad(inner_path)
print("saved:", outer_path)
print("saved:", inner_path)


## merge 後の確認用補助指標（現在の `.X` ベース）

merge 後の `.X` は混在スケールなので、全細胞に同じ意味で `total_counts` を解釈しない。
ここで計算する補助指標は **現在の `.X`（混在スケール）** に基づくものである。

```
matrix_sum            : 現在の .X の行和。
                        raw count では count sum、CPM/TPM では expression sum、
                        log normalized では log-expression sum。同じ意味ではない。
n_nonzero_genes       : 現在の .X で 0 でない遺伝子数。
pct_mt_current_matrix : 現在の .X に基づく mitochondrial fraction。
                        raw count 由来の mitochondrial fraction とは限らない。
```


In [ ]:
def add_current_matrix_metrics(adata):
    """現在の .X（混在スケール）に基づく補助指標を obs に付ける。.X は変更しない。"""
    X = adata.X
    Xc = X.tocsr() if sp.issparse(X) else np.asarray(X)

    matrix_sum = np.asarray(Xc.sum(axis=1)).ravel()
    if sp.issparse(Xc):
        n_nonzero = Xc.getnnz(axis=1)
    else:
        n_nonzero = (Xc != 0).sum(axis=1)

    gene_upper = pd.Index(adata.var_names).astype(str).str.upper()
    mt_mask = gene_upper.str.startswith("MT-").to_numpy()
    if mt_mask.any():
        mt_sum = np.asarray(Xc[:, mt_mask].sum(axis=1)).ravel()
    else:
        mt_sum = np.zeros(adata.n_obs)
    with np.errstate(divide="ignore", invalid="ignore"):
        pct_mt = np.where(matrix_sum > 0, 100.0 * mt_sum / matrix_sum, 0.0)

    adata.obs["matrix_sum"] = matrix_sum
    adata.obs["n_nonzero_genes"] = np.asarray(n_nonzero).ravel()
    adata.obs["pct_mt_current_matrix"] = pct_mt
    return adata


merged_outer = add_current_matrix_metrics(merged_outer)
merged_outer.obs[["qc_preprocessing_state", "matrix_sum", "n_nonzero_genes", "pct_mt_current_matrix"]].head()


## 可視化（preprocessing state / dataset 別）

In [ ]:
def sample_for_plot(adata):
    if adata.n_obs > MAX_PLOT:
        idx = np.sort(RNG.choice(adata.n_obs, size=MAX_PLOT, replace=False))
        return adata[idx].copy()
    return adata


obs = merged_outer.obs

# 1. dataset x Condition の細胞数（積み上げ棒）
ct = obs.groupby(["dataset_id", "Condition"], observed=True).size().unstack(fill_value=0)
ax = ct.plot(kind="bar", stacked=True, figsize=(max(6, 0.5 * len(ct)), 4))
ax.set_ylabel("n_cells")
ax.set_title("cells per dataset x Condition")
plt.tight_layout()
plt.savefig(PLOT_DIR / "merged_cells_per_dataset_condition.png", dpi=100, bbox_inches="tight")
plt.close("all")

# 2-4. preprocessing state / dataset 別に補助指標を violin（Scanpy）
plot_adata = sample_for_plot(merged_outer)
for key, fname in [
    ("matrix_sum", "merged_matrix_sum_by_dataset.png"),
    ("n_nonzero_genes", "merged_n_nonzero_genes_by_dataset.png"),
    ("pct_mt_current_matrix", "merged_pct_mt_current_matrix_by_dataset.png"),
]:
    try:
        sc.pl.violin(plot_adata, keys=key, groupby="dataset_id", rotation=90, show=False)
        plt.savefig(PLOT_DIR / fname, dpi=100, bbox_inches="tight")
    except Exception as e:
        print(f"  [warn] violin {key} failed: {e}")
    finally:
        plt.close("all")

print("saved plots ->", PLOT_DIR)


## summary CSV を保存

In [ ]:
obs = merged_outer.obs

# dataset 別に現在の .X で発現している遺伝子数（nonzero gene 数）を数える
n_genes_map = {}
for did in obs["dataset_id"].astype(str).unique():
    sub = merged_outer[obs["dataset_id"].astype(str).values == did]
    Xc = sub.X.tocsr() if sp.issparse(sub.X) else np.asarray(sub.X)
    if sp.issparse(Xc):
        gene_nnz = Xc.getnnz(axis=0)
    else:
        gene_nnz = (Xc != 0).sum(axis=0)
    n_genes_map[did] = int((np.asarray(gene_nnz).ravel() > 0).sum())

# merged_dataset_summary.csv
ds = (
    obs.groupby(["source_accession", "dataset_id", "qc_preprocessing_state"], observed=True)
    .agg(
        n_cells=("cell_uid", "size"),
        median_matrix_sum=("matrix_sum", "median"),
        median_n_nonzero_genes=("n_nonzero_genes", "median"),
        median_pct_mt_current_matrix=("pct_mt_current_matrix", "median"),
    )
    .reset_index()
)
ds["n_genes"] = ds["dataset_id"].astype(str).map(n_genes_map)
ds = ds[[
    "source_accession", "dataset_id", "qc_preprocessing_state",
    "n_cells", "n_genes", "median_matrix_sum",
    "median_n_nonzero_genes", "median_pct_mt_current_matrix",
]]
ds.to_csv(RESULTS_DIR / "merged_dataset_summary.csv", index=False)

# merged_condition_summary.csv
cond = (
    obs.groupby(["source_accession", "dataset_id", "Condition"], observed=True)
    .size().reset_index(name="n_cells")
)
cond.to_csv(RESULTS_DIR / "merged_condition_summary.csv", index=False)

# merged_final_summary.csv
final = pd.DataFrame([{
    "total_cells_merged": int(merged_outer.n_obs),
    "n_datasets": int(obs["dataset_id"].nunique()),
    "n_conditions": int(obs["Condition"].nunique()),
    "n_genes_outer": int(merged_outer.n_vars),
    "n_genes_inner": int(merged_inner.n_vars),
}])
final.to_csv(RESULTS_DIR / "merged_final_summary.csv", index=False)

print("saved summaries ->", RESULTS_DIR)
display(ds)
display(final)


## （参考）前処理を揃えるための関数例

以下は、merge 後の original-scale AnnData から **探索用の log-expression AnnData** を作るための関数例である。

- この 04d の標準出力は **original-scale merge** であり、この関数はデフォルトでは実行しない。
- 関数は必ず `adata.copy()` 相当の新しい AnnData を返し、入力の `merged` を破壊しない。
- 出力は探索的な可視化 / クラスタリング用であり、raw-count 行列ではない。scVI の count 入力には使わない。

今後必要になった時に、コメントを外して手動で使う。


In [ ]:
# def make_logexpr_copy_from_original_scale(
#     adata,
#     state_col="qc_preprocessing_state",
#     target_sum=1e4,
# ):
#     """
#     Return a new AnnData whose .X is converted to a rough common log-expression space.
#
#     This function does NOT modify the input adata.
#
#     Rules:
#       - raw_count_like: normalize_total(target_sum=1e4) then log1p
#       - cpm_tpm_like: normalize_total(target_sum=1e4) then log1p
#       - log_normalized_like: keep as is
#
#     Important:
#       This output is for exploratory visualization / clustering only.
#       It is not a raw-count matrix and should not be used as count input for scVI.
#     """
#     import anndata as ad
#
#     parts = []
#     for state in adata.obs[state_col].astype(str).unique():
#         sub = adata[adata.obs[state_col].astype(str) == state].copy()
#
#         if state in ["raw_count_like", "cpm_tpm_like"]:
#             sc.pp.normalize_total(sub, target_sum=target_sum)
#             sc.pp.log1p(sub)
#         elif state == "log_normalized_like":
#             pass
#         else:
#             print(f"WARNING: unknown preprocessing state: {state}; keeping .X as is")
#
#         parts.append(sub)
#
#     out = ad.concat(
#         parts,
#         join="outer",
#         merge="same",
#         label=None,
#         index_unique=None,
#     )
#     out.uns["made_from"] = "merged_qc_original_scale"
#     out.uns["logexpr_conversion_note"] = (
#         "raw_count_like and cpm_tpm_like were normalize_total+log1p; "
#         "log_normalized_like was kept as is."
#     )
#     return out
